# LeetCode 415 — Add Strings bằng Python

**Link đề:** https://leetcode.com/problems/add-strings/
**Độ khó:** Easy · **Chủ đề:** Math, String, Simulation

```python
class Solution:
    def addStrings(self, num1: str, num2: str) -> str:
        ...
```

## Mục tiêu học
1. Mô phỏng **phép cộng cột dọc** của tiểu học bằng code: cộng từ phải sang trái, có **số nhớ** (carry).
2. Đổi ký tự số ↔ giá trị số bằng `ord(ch) - ord('0')` và `chr(d + ord('0'))` — **không** dùng `int(num1)`.
3. Nắm kỹ thuật **hai con trỏ đi từ cuối** trên hai chuỗi khác độ dài, `divmod`, `zfill`, `zip_longest`.
4. Viết 4 lời giải: dùng `int` (chỉ để đối chiếu) → hai con trỏ + carry (chính) → đệm số 0 cho bằng độ dài → `zip_longest` trên chuỗi đảo.
5. Trace, chứng minh, Time/Space, lỗi hay gặp, biến thể (Add Binary, Add to Array-Form, nhân chuỗi).

## 1. Đề bài

Cho hai số nguyên **không âm** `num1`, `num2` biểu diễn dưới dạng **chuỗi**. Trả về **tổng** của chúng dưới dạng chuỗi.

**Ràng buộc quan trọng:** không được dùng thư viện số lớn (như `BigInteger`) và **không được đổi trực tiếp** input sang số nguyên (`int(num1)` bị cấm).

| `num1` | `num2` | Output |
|---|---|---|
| `"11"` | `"123"` | `"134"` |
| `"456"` | `"77"` | `"533"` |
| `"0"` | `"0"` | `"0"` |

**Constraints:**
- `1 <= num1.length, num2.length <= 10^4`
- Chỉ gồm chữ số; không có số 0 dẫn đầu (trừ chính số `0`).

Vì sao đề cấm `int()`? Trong nhiều ngôn ngữ, số 10^4 chữ số vượt xa kiểu số nguyên 64-bit. Bài này luyện cách **tự cộng số lớn** — nền tảng của thư viện số lớn.

## 2. Nền tảng: phép cộng cột dọc

```text
    4 5 6
  +   7 7
  -------
    5 3 3
```

Cộng **từ phải sang trái**:

| Cột | Chữ số 1 | Chữ số 2 | carry vào | Tổng | Viết | carry ra |
|---|:---:|:---:|:---:|:---:|:---:|:---:|
| đơn vị | 6 | 7 | 0 | 13 | 3 | 1 |
| chục | 5 | 7 | 1 | 13 | 3 | 1 |
| trăm | 4 | (không có = 0) | 1 | 5 | 5 | 0 |

- **Viết** = `tổng % 10`, **nhớ** = `tổng // 10`.
- Số ngắn hơn được coi như có số 0 ở bên trái.
- Hết cả hai số mà còn nhớ → viết thêm `1` ở đầu (ví dụ `"9" + "1" = "10"`).

In [1]:
for total in [13, 9, 10, 19]:
    print(f"tổng={total:>2} -> viết {total % 10}, nhớ {total // 10} | divmod: {divmod(total, 10)}")

tổng=13 -> viết 3, nhớ 1 | divmod: (1, 3)
tổng= 9 -> viết 9, nhớ 0 | divmod: (0, 9)
tổng=10 -> viết 0, nhớ 1 | divmod: (1, 0)
tổng=19 -> viết 9, nhớ 1 | divmod: (1, 9)


`divmod(a, b)` trả **cặp** `(a // b, a % b)` trong một lần gọi — tiện cho phép cộng: `carry, digit = divmod(total, 10)`.

### 2.1. Ký tự số ↔ giá trị số mà không dùng `int(chuỗi)`

Các ký tự `'0'..'9'` có mã Unicode liên tiếp `48..57`.
- `ord(ch) - ord('0')` → giá trị chữ số (`'7'` → 7).
- `chr(d + ord('0'))` → ký tự của chữ số `d` (7 → `'7'`).

(`int('7')` cho **một ký tự** thường được chấp nhận; cái bị cấm là đổi **cả chuỗi** `int(num1)`. Nhưng dùng `ord` cho chắc chắn và đúng tinh thần đề.)

In [2]:
print("ord('0') =", ord("0"), "| ord('7') =", ord("7"))
for ch in "0479":
    d = ord(ch) - ord("0")
    print(repr(ch), "->", d, "->", repr(chr(d + ord("0"))))

ord('0') = 48 | ord('7') = 55
'0' -> 0 -> '0'
'4' -> 4 -> '4'
'7' -> 7 -> '7'
'9' -> 9 -> '9'


### 2.2. Duyệt chuỗi từ cuối bằng chỉ số

Chỉ số cuối của chuỗi `s` là `len(s) - 1`. Đi lùi: `i -= 1` cho tới khi `i < 0`. (Python cũng có chỉ số âm `s[-1]`, nhưng với hai con trỏ ta dùng chỉ số dương cho rõ ràng.)

In [3]:
s = "456"
i = len(s) - 1
while i >= 0:
    print("i =", i, "->", s[i])
    i -= 1
print("i cuối cùng =", i)

i = 2 -> 6
i = 1 -> 5
i = 0 -> 4
i cuối cùng = -1


## 3. Cách trực giác (baseline) — dùng `int` (bị cấm, chỉ để đối chiếu)

Cách "ăn gian": `str(int(num1) + int(num2))`. Python có số nguyên vô hạn nên nó **chạy đúng** — vì vậy rất hữu ích làm **oracle** để kiểm tra lời giải tự viết. Nhưng nộp cách này là vi phạm đề.

In [4]:
def add_strings_builtin(num1: str, num2: str) -> str:
    # CHỈ để kiểm tra
    return str(int(num1) + int(num2))

print(add_strings_builtin("11", "123"), add_strings_builtin("456", "77"), add_strings_builtin("0", "0"))

134 533 0


### 3.1. Cách trực giác khác: đổi sang số bằng tay rồi cộng

Ta có thể tự đọc chuỗi thành số: `value = value * 10 + digit`. Không dùng `int(num1)`, nhưng **về bản chất vẫn là đổi toàn bộ sang số nguyên** → với 10^4 chữ số, ngôn ngữ khác sẽ tràn số; Python thì chạy được nhưng phép nhân số lớn mỗi bước tốn `O(độ dài)` → tổng `O(n²)`. Đây là **nút thắt**.

In [5]:
def parse_manual(s: str) -> int:
    value = 0
    for ch in s:
        value = value * 10 + (ord(ch) - ord("0"))
    return value

def add_strings_parse(num1: str, num2: str) -> str:
    return str(parse_manual(num1) + parse_manual(num2))

print(add_strings_parse("456", "77"))

533


## 4. Nút thắt và insight

**Nút thắt:** đổi cả chuỗi sang số là trái tinh thần đề và không mở rộng được sang ngôn ngữ có số nguyên giới hạn.

**Insight:** ta **không cần** giá trị của cả số. Mỗi cột chỉ cần: chữ số của `num1`, chữ số của `num2` ở cột đó, và `carry` (0 hoặc 1). Cộng từ phải sang trái, mỗi cột cho ra **một** chữ số kết quả.

Hai con trỏ `i` (cuối `num1`) và `j` (cuối `num2`) cùng đi lùi. Khi một chuỗi hết, coi chữ số của nó là `0`. Vòng lặp chạy khi **còn chữ số ở một trong hai chuỗi hoặc còn carry**:

```python
while i >= 0 or j >= 0 or carry:
```

Các chữ số kết quả sinh ra theo thứ tự **từ phải sang trái** → gom vào list rồi **đảo** ở cuối.

In [6]:
# Cột đơn vị của 456 + 77
num1, num2 = "456", "77"
i, j, carry = len(num1) - 1, len(num2) - 1, 0
d1 = ord(num1[i]) - ord("0")
d2 = ord(num2[j]) - ord("0")
carry, digit = divmod(d1 + d2 + carry, 10)
print("d1 =", d1, "d2 =", d2, "-> viết", digit, "nhớ", carry)

d1 = 6 d2 = 7 -> viết 3 nhớ 1


## 5. Lời giải chính — hai con trỏ từ cuối + carry

**Đoạn 1 — khởi tạo:** `i = len(num1) - 1`, `j = len(num2) - 1`, `carry = 0`, `out = []`.

**Đoạn 2 — vòng lặp:**
```python
while i >= 0 or j >= 0 or carry:
    d1 = ord(num1[i]) - 48 if i >= 0 else 0
    d2 = ord(num2[j]) - 48 if j >= 0 else 0
    carry, digit = divmod(d1 + d2 + carry, 10)
    out.append(chr(digit + 48))
    i -= 1
    j -= 1
```
(`48` chính là `ord('0')`.)

**Đoạn 3 — đảo và ghép:** `return "".join(reversed(out))`.

In [7]:
def add_strings_two_pointers(num1: str, num2: str) -> str:
    i, j = len(num1) - 1, len(num2) - 1
    carry = 0
    out = []
    while i >= 0 or j >= 0 or carry:
        d1 = ord(num1[i]) - ord("0") if i >= 0 else 0
        d2 = ord(num2[j]) - ord("0") if j >= 0 else 0
        carry, digit = divmod(d1 + d2 + carry, 10)
        out.append(chr(digit + ord("0")))
        i -= 1
        j -= 1
    return "".join(reversed(out))

print(add_strings_two_pointers("11", "123"))   # 134
print(add_strings_two_pointers("456", "77"))   # 533
print(add_strings_two_pointers("0", "0"))      # 0
print(add_strings_two_pointers("999", "1"))    # 1000

134
533
0
1000


## 6. TRACE từng bước

Trace `num1 = "999"`, `num2 = "1"` (có nhớ dây chuyền và chữ số mới ở đầu):

| Vòng | i | j | d1 | d2 | carry vào | tổng | viết | carry ra | out |
|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|---|
| 1 | 2 | 0 | 9 | 1 | 0 | 10 | 0 | 1 | `['0']` |
| 2 | 1 | -1 | 9 | 0 | 1 | 10 | 0 | 1 | `['0','0']` |
| 3 | 0 | -2 | 9 | 0 | 1 | 10 | 0 | 1 | `['0','0','0']` |
| 4 | -1 | -3 | 0 | 0 | 1 | 1 | 1 | 0 | `['0','0','0','1']` |

Dừng: `i < 0`, `j < 0`, `carry = 0`. Đảo → `"1000"`.

In [8]:
def trace_add_strings(num1: str, num2: str) -> str:
    i, j = len(num1) - 1, len(num2) - 1
    carry = 0
    out = []
    step = 1
    print(f"{num1} + {num2}")
    while i >= 0 or j >= 0 or carry:
        d1 = ord(num1[i]) - ord("0") if i >= 0 else 0
        d2 = ord(num2[j]) - ord("0") if j >= 0 else 0
        carry_in = carry
        carry, digit = divmod(d1 + d2 + carry, 10)
        out.append(chr(digit + ord("0")))
        print(f"  vòng {step}: i={i:>2}, j={j:>2} | {d1}+{d2}+{carry_in}={d1 + d2 + carry_in:>2} "
              f"-> viết {digit}, nhớ {carry} | out={out}")
        i -= 1
        j -= 1
        step += 1
    res = "".join(reversed(out))
    print("  đảo ->", res)
    return res

trace_add_strings("999", "1")
print()
trace_add_strings("456", "77")

999 + 1
  vòng 1: i= 2, j= 0 | 9+1+0=10 -> viết 0, nhớ 1 | out=['0']
  vòng 2: i= 1, j=-1 | 9+0+1=10 -> viết 0, nhớ 1 | out=['0', '0']
  vòng 3: i= 0, j=-2 | 9+0+1=10 -> viết 0, nhớ 1 | out=['0', '0', '0']
  vòng 4: i=-1, j=-3 | 0+0+1= 1 -> viết 1, nhớ 0 | out=['0', '0', '0', '1']
  đảo -> 1000

456 + 77
  vòng 1: i= 2, j= 1 | 6+7+0=13 -> viết 3, nhớ 1 | out=['3']
  vòng 2: i= 1, j= 0 | 5+7+1=13 -> viết 3, nhớ 1 | out=['3', '3']
  vòng 3: i= 0, j=-1 | 4+0+1= 5 -> viết 5, nhớ 0 | out=['3', '3', '5']
  đảo -> 533


'533'

## 7. Các lời giải khác

### 7.1. Cách 3 — đệm số 0 cho bằng độ dài (`zfill`)

`s.zfill(w)` thêm số 0 bên **trái** cho đủ `w` ký tự. Sau khi hai chuỗi dài bằng nhau, chỉ cần **một** chỉ số chạy từ cuối về đầu. Đổi lại tốn thêm bộ nhớ cho chuỗi đệm.

In [9]:
print("77".zfill(3), "456".zfill(3), "5".zfill(4))

def add_strings_padding(num1: str, num2: str) -> str:
    n = max(len(num1), len(num2))
    a, b = num1.zfill(n), num2.zfill(n)
    carry = 0
    out = []
    for k in range(n - 1, -1, -1):
        carry, digit = divmod((ord(a[k]) - 48) + (ord(b[k]) - 48) + carry, 10)
        out.append(chr(digit + 48))
    if carry:
        out.append("1")
    return "".join(reversed(out))

print(add_strings_padding("11", "123"), add_strings_padding("999", "1"))

077 456 0005
134 1000


### 7.2. Cách 4 — `zip_longest` trên chuỗi đảo

`itertools.zip_longest(a, b, fillvalue="0")` ghép từng cặp phần tử; chuỗi nào hết trước được điền `"0"`. Đảo hai chuỗi trước (`[::-1]`) để đi từ hàng đơn vị.

In [10]:
from itertools import zip_longest

print(list(zip_longest("654", "77", fillvalue="0")))

def add_strings_zip(num1: str, num2: str) -> str:
    carry = 0
    out = []
    for c1, c2 in zip_longest(num1[::-1], num2[::-1], fillvalue="0"):
        carry, digit = divmod(ord(c1) + ord(c2) - 2 * ord("0") + carry, 10)
        out.append(str(digit))
    if carry:
        out.append("1")
    return "".join(out[::-1])

print(add_strings_zip("11", "123"), add_strings_zip("456", "77"), add_strings_zip("999", "1"))

[('6', '7'), ('5', '7'), ('4', '0')]
134 533 1000


`str(digit)` ở đây đổi **một chữ số** (0..9) sang ký tự — hợp lệ vì ta không đổi input. Cũng có thể dùng `"0123456789"[digit]`.

## 8. Code nộp LeetCode

Bản nộp là hai con trỏ + carry (dùng `48 = ord('0')` cho gọn). Không import, không `int(chuỗi)`.

In [11]:
class Solution:
    def addStrings(self, num1: str, num2: str) -> str:
        i, j = len(num1) - 1, len(num2) - 1
        carry = 0
        out = []
        while i >= 0 or j >= 0 or carry:
            d1 = ord(num1[i]) - 48 if i >= 0 else 0
            d2 = ord(num2[j]) - 48 if j >= 0 else 0
            carry, digit = divmod(d1 + d2 + carry, 10)
            out.append(chr(digit + 48))
            i -= 1
            j -= 1
        return "".join(reversed(out))


sol = Solution()
print(sol.addStrings("11", "123"), sol.addStrings("456", "77"), sol.addStrings("0", "0"))

134 533 0


## 9. Vì sao đúng? (invariant)

Gọi `k` là số vòng đã chạy. **Bất biến:** chuỗi `reversed(out)` (dài `k`) là `k` chữ số thấp nhất của tổng, và

```text
(k chữ số cuối của num1) + (k chữ số cuối của num2) = carry · 10^k + giá_trị(reversed(out))
```

- `k = 0`: hai vế bằng 0 → đúng.
- Một vòng: cộng chữ số cột `k` của hai số (hoặc 0 nếu hết) với `carry` cũ → `tổng = 10·carry_mới + digit`. Thêm `digit` vào vị trí `10^k` → bất biến giữ cho `k + 1`.
- Vì `d1, d2 <= 9`, `carry <= 1` → `tổng <= 19` → carry luôn là 0 hoặc 1.
- Dừng khi hết chữ số **và** `carry = 0` → vế trái là toàn bộ `num1 + num2`, vế phải là `giá_trị(kết quả)`. ∎

Không có số 0 dẫn đầu: chữ số cuối cùng được thêm hoặc là chữ số của cột cao nhất (khác 0 trừ trường hợp `"0"+"0"`), hoặc là carry `1`.

## 10. Độ phức tạp

Gọi `n = len(num1)`, `m = len(num2)`.

| Cách | Time | Space phụ |
|---|:---:|:---:|
| `int()` (cấm) | phụ thuộc thư viện; thường `O(n + m)`–`O((n+m)^2)` | `O(n + m)` |
| Tự đọc số rồi cộng | `O(n² + m²)` (nhân số lớn mỗi bước) | `O(n + m)` |
| **Hai con trỏ + carry** | `O(max(n, m))` | `O(max(n, m))` cho kết quả |
| Đệm `zfill` | `O(max(n, m))` | `O(max(n, m))` thêm 2 chuỗi đệm |
| `zip_longest` | `O(max(n, m))` | `O(n + m)` cho chuỗi đảo |

Kết quả có tối đa `max(n, m) + 1` chữ số, nên `O(max(n, m))` là tối ưu.

## 11. Bộ test

In [12]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name:<30} -> {actual if len(actual) < 30 else actual[:12] + '...(' + str(len(actual)) + ')'}")

sol = Solution()
cases = [
    ("đề 1", "11", "123", "134"),
    ("đề 2", "456", "77", "533"),
    ("đề 3", "0", "0", "0"),
    ("nhớ dây chuyền", "999", "1", "1000"),
    ("một chữ số có nhớ", "9", "9", "18"),
    ("khác độ dài nhiều", "1", "99999", "100000"),
    ("cộng với 0", "12345", "0", "12345"),
    ("không nhớ", "1234", "4321", "5555"),
    ("10^4 chữ số 9 + 1", "9" * 10 ** 4, "1", "1" + "0" * 10 ** 4),
]
for name, a, b_, exp in cases:
    check(name, sol.addStrings(a, b_), exp)

PASS  đề 1                           -> 134
PASS  đề 2                           -> 533
PASS  đề 3                           -> 0
PASS  nhớ dây chuyền                 -> 1000
PASS  một chữ số có nhớ              -> 18
PASS  khác độ dài nhiều              -> 100000
PASS  cộng với 0                     -> 12345
PASS  không nhớ                      -> 5555
PASS  10^4 chữ số 9 + 1              -> 100000000000...(10001)


In [13]:
import random
rng = random.Random(415)

def rand_num(rng, length):
    if length == 1:
        return str(rng.randint(0, 9))
    return str(rng.randint(1, 9)) + "".join(str(rng.randint(0, 9)) for _ in range(length - 1))

funcs = [add_strings_two_pointers, add_strings_padding, add_strings_zip, Solution().addStrings]
for _ in range(1500):
    a = rand_num(rng, rng.randint(1, 40))
    b_ = rand_num(rng, rng.randint(1, 40))
    exp = add_strings_builtin(a, b_)
    for f in funcs:
        assert f(a, b_) == exp, (f.__name__, a, b_)
print("1500 cặp ngẫu nhiên: 4 cách khớp oracle int() -> PASS")

1500 cặp ngẫu nhiên: 4 cách khớp oracle int() -> PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1: Quên carry cuối cùng
```python
while i >= 0 or j >= 0:      # thiếu "or carry"
```
`"999" + "1"` → `"000"` thay vì `"1000"`.

### Lỗi 2: Quên đảo kết quả
Chữ số được thêm từ hàng đơn vị → ghép thẳng ra `"335"` thay vì `"533"`.

In [14]:
def wrong_no_final_carry(num1, num2):
    i, j, carry, out = len(num1) - 1, len(num2) - 1, 0, []
    while i >= 0 or j >= 0:
        d1 = ord(num1[i]) - 48 if i >= 0 else 0
        d2 = ord(num2[j]) - 48 if j >= 0 else 0
        carry, digit = divmod(d1 + d2 + carry, 10)
        out.append(chr(digit + 48)); i -= 1; j -= 1
    return "".join(reversed(out))

def wrong_no_reverse(num1, num2):
    i, j, carry, out = len(num1) - 1, len(num2) - 1, 0, []
    while i >= 0 or j >= 0 or carry:
        d1 = ord(num1[i]) - 48 if i >= 0 else 0
        d2 = ord(num2[j]) - 48 if j >= 0 else 0
        carry, digit = divmod(d1 + d2 + carry, 10)
        out.append(chr(digit + 48)); i -= 1; j -= 1
    return "".join(out)

print("lỗi 1:", wrong_no_final_carry("999", "1"), "(đúng 1000)")
print("lỗi 2:", wrong_no_reverse("456", "77"), "(đúng 533)")

lỗi 1: 000 (đúng 1000)
lỗi 2: 335 (đúng 533)


### Lỗi 3: Cộng ký tự thay vì cộng số
`"4" + "7"` trong Python là **nối chuỗi** `"47"`, không phải 11. Phải đổi sang số trước.

### Lỗi 4: Đọc `num1[i]` khi `i` đã âm
Python **không báo lỗi** với `num1[-1]` (nó lấy ký tự cuối!) → kết quả sai âm thầm. Phải có `if i >= 0 else 0`.

### Lỗi 5: Nối chuỗi `res = ch + res` trong vòng lặp
Mỗi lần tạo chuỗi mới dài dần → `O(n²)`. Dùng list + `join`.

In [15]:
print("'4' + '7' =", "4" + "7")

def wrong_negative_index(num1, num2):
    out, carry = [], 0
    for k in range(max(len(num1), len(num2))):
        i, j = len(num1) - 1 - k, len(num2) - 1 - k
        carry, digit = divmod((ord(num1[i]) - 48) + (ord(num2[j]) - 48) + carry, 10)   # thiếu kiểm tra i, j >= 0
        out.append(chr(digit + 48))
    if carry:
        out.append("1")
    return "".join(reversed(out))

print("lỗi 4: 456 + 77 ->", wrong_negative_index("456", "77"), "(đúng 533) — num2[-1] lấy nhầm '7'")

'4' + '7' = 47
lỗi 4: 456 + 77 -> 1233 (đúng 533) — num2[-1] lấy nhầm '7'


## 13. Mẹo nhận dạng pattern

- "Số rất lớn biểu diễn bằng **chuỗi / mảng / danh sách liên kết**" + "cộng/trừ/nhân" → mô phỏng cột dọc với **carry**.
- Hai dãy khác độ dài, xử lý từ cuối → hai con trỏ lùi + "hết thì coi là 0" + `while i >= 0 or j >= 0 or carry`.
- Kết quả sinh ngược → `append` rồi đảo một lần.

**Họ bài cùng template:** 2 Add Two Numbers (linked list), 67 Add Binary, 66 Plus One, 989 Add to Array-Form of Integer, 43 Multiply Strings (Medium).

## 14. Biến thể

1. **LeetCode 67 — Add Binary:** cùng code, đổi cơ số 10 → 2.
2. **Cộng cơ số bất kỳ** — tham số `base`.
3. **LeetCode 43 — Multiply Strings:** nhân từng cặp chữ số vào mảng kết quả `res[i + j + 1]`.

In [16]:
def add_base(num1: str, num2: str, base: int) -> str:
    i, j, carry, out = len(num1) - 1, len(num2) - 1, 0, []
    while i >= 0 or j >= 0 or carry:
        d1 = ord(num1[i]) - 48 if i >= 0 else 0
        d2 = ord(num2[j]) - 48 if j >= 0 else 0
        carry, digit = divmod(d1 + d2 + carry, base)
        out.append(chr(digit + 48)); i -= 1; j -= 1
    return "".join(reversed(out))

def multiply_strings(num1: str, num2: str) -> str:
    res = [0] * (len(num1) + len(num2))
    for i in range(len(num1) - 1, -1, -1):
        for j in range(len(num2) - 1, -1, -1):
            p = (ord(num1[i]) - 48) * (ord(num2[j]) - 48) + res[i + j + 1]
            res[i + j + 1] = p % 10
            res[i + j] += p // 10
    s = "".join(map(str, res)).lstrip("0")
    return s or "0"

print("Add Binary 1010 + 1011 =", add_base("1010", "1011", 2))
print("123 * 456 =", multiply_strings("123", "456"), "| kiểm:", 123 * 456)

Add Binary 1010 + 1011 = 10101
123 * 456 = 56088 | kiểm: 56088


## 15. Bài tập (TODO)

**Bài 1 (tính tay):** Lập bảng trace cho `"58" + "967"`.

**Bài 2 (đoán output):** `wrong_no_final_carry("5", "5")` trả gì?

**Bài 3 (điền chỗ trống):** hoàn thiện hai con trỏ.

In [17]:
def exercise_add_strings(num1: str, num2: str) -> str:
    i, j, carry, out = len(num1) - 1, len(num2) - 1, 0, []
    # TODO 1: điều kiện while (3 vế)
    # TODO 2: lấy d1, d2 (0 nếu hết)
    # TODO 3: divmod, append, lùi i, j
    # TODO 4: đảo và ghép
    pass

# assert exercise_add_strings("456", "77") == "533"
# assert exercise_add_strings("999", "1") == "1000"

**Bài 4 (tự viết):** `exercise_add_binary(a, b)` cộng hai chuỗi nhị phân (LeetCode 67).

**Bài 5 (biến thể):** `exercise_add_array_form(num, k)` — `num` là list chữ số (ví dụ `[1,2,0,0]`), `k` là số nguyên; trả list chữ số của `num + k` (LeetCode 989). Gợi ý: coi `k` như carry ban đầu.

**Bài 6 (mini challenge):** `exercise_sum_many(nums)` cộng **danh sách** nhiều chuỗi số (≥ 1 chuỗi) không dùng `int(chuỗi)`.

In [18]:
def exercise_add_binary(a: str, b: str) -> str:
    # TODO
    pass

def exercise_add_array_form(num: list, k: int) -> list:
    # TODO
    pass

def exercise_sum_many(nums: list) -> str:
    # TODO
    pass

# assert exercise_add_binary("11", "1") == "100"
# assert exercise_add_array_form([2, 7, 4], 181) == [4, 5, 5]
# assert exercise_sum_many(["1", "99", "900"]) == "1000"

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Cột đơn vị: 8 + 7 | 15 → viết 5 nhớ 1 | Kết quả có 4 chữ số? |
| 2 | Vòng lặp dừng khi nào? | Thiếu `or carry` | Chỉ còn chữ số đơn vị |
| 3 | Theo mục 5 | `while i >= 0 or j >= 0 or carry` | `carry, digit = divmod(d1 + d2 + carry, 10)` |
| 4 | Đổi 10 thành 2 | `divmod(..., 2)` | Dùng lại `add_base` hoặc tự viết |
| 5 | `k` = carry khởi đầu | `while i >= 0 or k` | `k, d = divmod(num[i] + k, 10)` (nếu `i < 0` thì `num[i]` coi là 0) |
| 6 | Cộng dồn từng chuỗi | Gọi lời giải chính trong vòng lặp | `total = "0"; for s in nums: total = add(total, s)` |

## 17. Đáp án tham khảo

- **Bài 1:** `8+7=15` → viết 5 nhớ 1; `5+6+1=12` → viết 2 nhớ 1; `0+9+1=10` → viết 0 nhớ 1; còn nhớ → viết 1. Kết quả **`"1025"`**.
- **Bài 2:** `5+5=10` → viết 0 nhớ 1, nhưng vòng dừng → **`"0"`** (sai, đúng là `"10"`).

In [19]:
trace_add_strings("58", "967")
print("Bài 2:", wrong_no_final_carry("5", "5"))

58 + 967
  vòng 1: i= 1, j= 2 | 8+7+0=15 -> viết 5, nhớ 1 | out=['5']
  vòng 2: i= 0, j= 1 | 5+6+1=12 -> viết 2, nhớ 1 | out=['5', '2']
  vòng 3: i=-1, j= 0 | 0+9+1=10 -> viết 0, nhớ 1 | out=['5', '2', '0']
  vòng 4: i=-2, j=-1 | 0+0+1= 1 -> viết 1, nhớ 0 | out=['5', '2', '0', '1']
  đảo -> 1025
Bài 2: 0


In [20]:
def answer_add_strings(num1: str, num2: str) -> str:
    i, j, carry, out = len(num1) - 1, len(num2) - 1, 0, []
    while i >= 0 or j >= 0 or carry:
        d1 = ord(num1[i]) - 48 if i >= 0 else 0
        d2 = ord(num2[j]) - 48 if j >= 0 else 0
        carry, digit = divmod(d1 + d2 + carry, 10)
        out.append(chr(digit + 48))
        i -= 1
        j -= 1
    return "".join(reversed(out))

def answer_add_binary(a: str, b: str) -> str:
    i, j, carry, out = len(a) - 1, len(b) - 1, 0, []
    while i >= 0 or j >= 0 or carry:
        total = carry + (ord(a[i]) - 48 if i >= 0 else 0) + (ord(b[j]) - 48 if j >= 0 else 0)
        carry, bit = divmod(total, 2)
        out.append(str(bit))
        i -= 1
        j -= 1
    return "".join(reversed(out))

def answer_add_array_form(num: list, k: int) -> list:
    out = []
    i = len(num) - 1
    while i >= 0 or k:
        k, d = divmod((num[i] if i >= 0 else 0) + k, 10)
        out.append(d)
        i -= 1
    return out[::-1] or [0]

def answer_sum_many(nums: list) -> str:
    total = "0"
    for s in nums:
        total = answer_add_strings(total, s)
    return total

assert answer_add_strings("456", "77") == "533" and answer_add_strings("999", "1") == "1000"
assert answer_add_binary("11", "1") == "100" and answer_add_binary("1010", "1011") == "10101"
assert answer_add_array_form([2, 7, 4], 181) == [4, 5, 5] and answer_add_array_form([9, 9], 1) == [1, 0, 0]
assert answer_add_array_form([0], 0) == [0]
assert answer_sum_many(["1", "99", "900"]) == "1000" and answer_sum_many(["0"]) == "0"
print("Đáp án bài 3–6 PASS")

Đáp án bài 3–6 PASS


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def addStrings(self, num1: str, num2: str) -> str:
        i, j, carry, out = len(num1) - 1, len(num2) - 1, 0, []
        while i >= 0 or j >= 0 or carry:                 # còn chữ số HOẶC còn nhớ
            d1 = ord(num1[i]) - 48 if i >= 0 else 0
            d2 = ord(num2[j]) - 48 if j >= 0 else 0
            carry, digit = divmod(d1 + d2 + carry, 10)
            out.append(chr(digit + 48))
            i -= 1; j -= 1
        return "".join(reversed(out))
```

**Checklist:**
- [ ] Không `int(num1)` / `int(num2)`.
- [ ] Điều kiện vòng có `or carry`.
- [ ] Chỉ số âm → coi chữ số là 0 (Python `s[-1]` không báo lỗi!).
- [ ] Đảo kết quả trước khi ghép; dùng list + `join`.
- [ ] `"0" + "0"` → `"0"`.
- [ ] Time `O(max(n, m))`, Space `O(max(n, m))`.